In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
# EMA CROSSOVER v2 - HIGH-FREQUENCY TRADE GENERATION
# Improvements: trailing ATR stop, pullback re-entry, ADX filter, multi-asset

TICKERS = ['QQQ', 'SPY', 'NVDA', 'AMD', 'MSFT', 'AMZN', 'META', 'AAPL']
PRIMARY_TICKER = 'QQQ'
START_DATE = '2018-01-01'

all_data = download_multi(TICKERS, START_DATE)
for t, df in all_data.items():
    print(f"  {t}: {len(df)} bars")

stock_data = all_data[PRIMARY_TICKER]
TICKER = PRIMARY_TICKER
print(f"\nPrimary: {TICKER} ({len(stock_data)} bars)")

In [ ]:
# TECHNICAL ANALYSIS INDICATORS
close_arr = stock_data['Close'].values.astype(float)
high_arr  = stock_data['High'].values.astype(float)
low_arr   = stock_data['Low'].values.astype(float)
volume_arr = stock_data['Volume'].values.astype(float)
idx = stock_data.index

SMA_20 = talib.SMA(close_arr, 20); SMA_50 = talib.SMA(close_arr, 50)
EMA_12 = talib.EMA(close_arr, 12); EMA_26 = talib.EMA(close_arr, 26)
RSI_14 = talib.RSI(close_arr, 14)
ATR_14 = talib.ATR(high_arr, low_arr, close_arr, 14)
ADX_14 = talib.ADX(high_arr, low_arr, close_arr, 14)

indicators_df = pd.DataFrame({
    'Close': close_arr, 'SMA_20': SMA_20, 'SMA_50': SMA_50,
    'EMA_12': EMA_12, 'EMA_26': EMA_26,
    'RSI_14': RSI_14, 'ATR_14': ATR_14, 'ADX_14': ADX_14
}, index=idx)
print("Indicators computed.")
indicators_df.tail(5)

In [ ]:
# PREPARE PRICE SERIES
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

def get_ohlcv(df):
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy(); df.columns = [c[0] for c in df.columns]
    return (df['Close'].astype(float), df['High'].astype(float),
            df['Low'].astype(float), df['Volume'].astype(float))

close_s = stock_data['Close'].astype(float).squeeze(); close_s.name = 'price'
high_s  = stock_data['High'].astype(float).squeeze()
low_s   = stock_data['Low'].astype(float).squeeze()
vol_s   = stock_data['Volume'].astype(float).squeeze()

TRAIN_RATIO = 0.60
split_idx = int(len(close_s) * TRAIN_RATIO)
train_close = close_s.iloc[:split_idx].copy()
val_close   = close_s.iloc[split_idx:].copy()
train_high  = high_s.iloc[:split_idx].copy()
train_low   = low_s.iloc[:split_idx].copy()
train_vol   = vol_s.iloc[:split_idx].copy()

print(f"Train: {train_close.index[0].date()} to {train_close.index[-1].date()} ({len(train_close)} bars)")
print(f"Val:   {val_close.index[0].date()} to {val_close.index[-1].date()} ({len(val_close)} bars)")

# EMA Crossover v2 - High-Frequency Trade Generation

## What's New vs v1
| Feature | v1 | v2 |
|---|---|---|
| **Exit** | Wait for EMA cross-back | Trailing ATR stop (faster exits) |
| **Re-entry** | Only on new crossover | Pullback to fast EMA during uptrend |
| **Filter** | SMA trend filter | ADX >= 20 trend confirmation |
| **Assets** | Single ticker | 8-ticker scanning |

## Signal Logic (v2)
1. **Entry (crossover)**: Fast EMA crosses above Slow EMA AND ADX >= threshold
2. **Entry (pullback)**: In uptrend (fast > slow), price pulls back near fast EMA then bounces
3. **Exit**: Trailing stop at trail_atr * ATR below highest close since entry
4. All signals use previous bar indicators (anti-lookahead)

In [ ]:
# v2 SIGNAL ENGINE + PARAMETER RANGES

TRAIL_ATR_MULT = 2.0; ADX_MIN = 20; PULLBACK_ENABLED = True; PULLBACK_ATR_MULT = 0.5

def generate_v2_signals(close, high, low, volume, fast_ema, slow_ema,
                        trail_atr_mult=TRAIL_ATR_MULT, adx_min=ADX_MIN,
                        pullback=PULLBACK_ENABLED, pullback_atr_mult=PULLBACK_ATR_MULT,
                        atr_period=14):
    c = close.values.astype(float); h = high.values.astype(float)
    l = low.values.astype(float); v = volume.values.astype(float)
    idx = close.index; n = len(c)

    fast_vals = talib.EMA(c, timeperiod=fast_ema)
    slow_vals = talib.EMA(c, timeperiod=slow_ema)
    atr_vals  = talib.ATR(h, l, c, timeperiod=atr_period)
    adx_vals  = talib.ADX(h, l, c, timeperiod=atr_period)

    entries = np.zeros(n, dtype=bool); exits = np.zeros(n, dtype=bool)
    in_trade = False; highest = 0.0; stop = 0.0

    for i in range(max(slow_ema, atr_period) + 2, n):
        if np.isnan(fast_vals[i]) or np.isnan(slow_vals[i]) or np.isnan(atr_vals[i]):
            continue
        pf = fast_vals[i-1]; ps = slow_vals[i-1]; pa = atr_vals[i-1]
        padx = adx_vals[i-1] if not np.isnan(adx_vals[i-1]) else 0
        pc = c[i-1]; cc_ = c[i]
        p2f = fast_vals[i-2] if i >= 2 else np.nan
        p2s = slow_vals[i-2] if i >= 2 else np.nan

        if in_trade:
            if cc_ > highest:
                highest = cc_; stop = highest - trail_atr_mult * pa
            if cc_ < stop:
                exits[i] = True; in_trade = False; continue
            if not np.isnan(p2f) and p2f >= p2s and pf < ps:
                exits[i] = True; in_trade = False; continue
        else:
            if not np.isnan(p2f) and p2f <= p2s and pf > ps and padx >= adx_min:
                entries[i] = True; in_trade = True
                highest = cc_; stop = cc_ - trail_atr_mult * pa; continue
            if pullback and pf > ps and padx >= adx_min:
                dist = abs(pc - pf)
                if dist <= pullback_atr_mult * pa and cc_ > pc:
                    entries[i] = True; in_trade = True
                    highest = cc_; stop = cc_ - trail_atr_mult * pa; continue

    return pd.Series(entries, index=idx, dtype=bool), pd.Series(exits, index=idx, dtype=bool)


def generate_v1_signals(close, fast_ema, slow_ema, trend_filter):
    c = close.values.astype(float); idx = close.index
    fv = pd.Series(talib.EMA(c, timeperiod=fast_ema), index=idx)
    sv = pd.Series(talib.EMA(c, timeperiod=slow_ema), index=idx)
    tv = pd.Series(talib.SMA(c, timeperiod=trend_filter), index=idx)
    cs = pd.Series(c, index=idx)
    e_raw = ((fv.shift(1) <= sv.shift(1)) & (fv > sv) & (cs > tv))
    x_raw = ((fv.shift(1) >= sv.shift(1)) & (fv < sv))
    entries = pd.Series(np.where(e_raw.shift(1).isna(), False, e_raw.shift(1)), index=idx, dtype=bool)
    exits   = pd.Series(np.where(x_raw.shift(1).isna(), False, x_raw.shift(1)), index=idx, dtype=bool)
    return entries, exits

# Parameter Ranges
fast_ema_range  = list(range(5, 31, 1))    # 26 values
slow_ema_range  = list(range(20, 81, 2))   # 31 values
trail_atr_range = [1.5, 2.0, 2.5, 3.0]    # 4 values
adx_min_range   = [15, 20, 25]             # 3 values

ema_v2_combinations = []
for fast, slow, trail, adx in product(fast_ema_range, slow_ema_range, trail_atr_range, adx_min_range):
    if slow > fast + 5:
        ema_v2_combinations.append((fast, slow, trail, adx))

print(f"Fast EMA: {fast_ema_range[0]}-{fast_ema_range[-1]} ({len(fast_ema_range)})")
print(f"Slow EMA: {slow_ema_range[0]}-{slow_ema_range[-1]} ({len(slow_ema_range)})")
print(f"Trail ATR: {trail_atr_range}")
print(f"ADX min: {adx_min_range}")
print(f"\nTotal combos: {len(ema_v2_combinations)}")
print(f"\nFirst 10:")
for i, (f, s, t, a) in enumerate(ema_v2_combinations[:10], 1):
    print(f"  {i:2d}. Fast={f:2d} Slow={s:2d} Trail={t:.1f} ADX>={a}")

In [ ]:
# Initialize Results Collection
grid_search_results = []
print(f"EMA v2 Results Initialized | Testing {len(ema_v2_combinations)} combos")

In [ ]:
# EMA CROSSOVER v2 GRID SEARCH - TRAINING DATA

print("INITIATING EMA CROSSOVER v2 GRID SEARCH")
print("=" * 70)
print(f"Training: {train_close.index[0].date()} to {train_close.index[-1].date()}")
print(f"Capital: $100,000 | Fees: 0.05% | Slippage: 0.05%")
print("=" * 70)

total_combos = len(ema_v2_combinations)
grid_search_results = []; successful = 0; failed = 0; skipped = 0
years = max((train_close.index[-1] - train_close.index[0]).days / 365.25, 1e-9)

for combo_num, (fast, slow, trail, adx) in enumerate(ema_v2_combinations, 1):
    try:
        entries, exits = generate_v2_signals(train_close, train_high, train_low, train_vol,
                                             fast_ema=fast, slow_ema=slow,
                                             trail_atr_mult=trail, adx_min=adx)
        pf = vbt.Portfolio.from_signals(close=train_close, entries=entries, exits=exits,
                                        init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        trades = pf.trades; total_trades = len(trades)
        if total_trades < 5:
            skipped += 1
            continue

        trades_per_year = total_trades / years
        total_return = float(pf.total_return())
        ann_ret = float(pf.annualized_return(freq='D'))
        max_dd = float(pf.max_drawdown())
        vol = float(pf.annualized_volatility(freq='D'))
        sharpe = float(pf.sharpe_ratio(freq='D'))
        sortino = float(pf.sortino_ratio(freq='D'))
        calmar = ann_ret / abs(max_dd) if abs(max_dd) > 1e-9 else np.nan

        wr = np.nan; pf_ratio = np.nan; exp = 0.0
        aw = 0.0; al = 0.0; lw = 0.0; ll = 0.0; ws = np.nan; ls = np.nan; sqn = np.nan

        tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
        if tr.size > 0:
            pos = tr[tr > 0]; neg = tr[tr < 0]
            wr = (len(pos)/len(tr))*100 if len(tr) else np.nan
            g = pos.sum() if len(pos) else 0; l_ = abs(neg.sum()) if len(neg) else 0
            pf_ratio = g/l_ if l_ > 0 else np.inf
            exp = float(tr.mean())
            aw = float(pos.mean()) if len(pos) else 0
            al = float(abs(neg.mean())) if len(neg) else 0
            lw = float(pos.max()) if len(pos) else 0
            ll = float(neg.min()) if len(neg) else 0
            sqn = float(tr.mean()/tr.std()*np.sqrt(len(tr))) if tr.std() > 0 else np.nan
            try:
                ws = int(trades.winning_streak()); ls = int(trades.losing_streak())
            except:
                pass

        rets = pf.returns(); cum = (1+rets).cumprod(); pk = cum.cummax(); dd = (cum-pk)/pk
        ui = float(np.sqrt((dd.pow(2)).mean())) if len(dd) else np.nan
        pr = (aw/al) if al > 0 else np.inf
        rv = rets.values
        om = float(rv[rv>0].sum()/abs(rv[rv<0].sum())) if abs(rv[rv<0].sum()) > 0 else np.inf
        rf = total_return/abs(max_dd) if abs(max_dd) > 1e-9 else np.nan
        gp = float(rv.sum()/abs(rv[rv<0].sum())) if abs(rv[rv<0].sum()) > 0 else np.inf
        si = rf/ui if ui and ui > 0 else np.nan

        grid_search_results.append({
            "fast_ema": fast, "slow_ema": slow, "trail_atr_mult": trail, "adx_min": adx,
            "total_return": total_return, "annualized_return": ann_ret,
            "sharpe_ratio": sharpe, "sortino_ratio": sortino, "calmar_ratio": calmar,
            "omega_ratio": om, "max_drawdown": max_dd, "volatility": vol,
            "ulcer_index": ui, "win_rate": wr, "total_trades": total_trades,
            "trades_per_year": trades_per_year, "expectancy": exp,
            "profit_factor": pf_ratio, "sqn": sqn, "payoff_ratio": pr,
            "largest_win": lw, "largest_loss": ll, "avg_win_amount": aw, "avg_loss_amount": al,
            "winning_streak": ws, "losing_streak": ls,
            "recovery_factor": rf, "gain_to_pain_ratio": gp, "serenity_index": si
        })
        successful += 1
    except:
        failed += 1

    if combo_num % 500 == 0:
        print(f"Progress: {combo_num}/{total_combos} ({combo_num/total_combos*100:.1f}%) | Done: {successful} | Skip: {skipped}")

print("\n" + "=" * 70)
print("GRID SEARCH COMPLETE")
print(f"Tested: {total_combos} | OK: {successful} | Skip: {skipped} | Fail: {failed}")

if successful > 0:
    results_df = pd.DataFrame(grid_search_results)
    top_10 = results_df.nlargest(10, 'sharpe_ratio')
    print("\nTOP 10 BY SHARPE:")
    for rank, (_, row) in enumerate(top_10.iterrows(), 1):
        print(f"  #{rank} F={int(row['fast_ema'])} S={int(row['slow_ema'])} "
              f"T={row['trail_atr_mult']:.1f} ADX>={int(row['adx_min'])} | "
              f"SR={row['sharpe_ratio']:.3f} Ret={row['total_return']:.2%} "
              f"DD={row['max_drawdown']:.2%} Trades={int(row['total_trades'])} ({row['trades_per_year']:.1f}/yr)")

In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION
results_df = pd.DataFrame(grid_search_results)
if results_df.empty:
    print("No results.")
else:
    print("=" * 90)
    print("TOP 5 - OUT-OF-SAMPLE VALIDATION")
    print("=" * 90)
    val_high = high_s.iloc[split_idx:]; val_low = low_s.iloc[split_idx:]; val_vol = vol_s.iloc[split_idx:]
    top_5 = results_df.nlargest(5, 'sharpe_ratio'); oos_results = []
    for _, st in top_5.iterrows():
        fe=int(st['fast_ema']); se=int(st['slow_ema']); tr_=float(st['trail_atr_mult']); ad=int(st['adx_min'])
        e, x = generate_v2_signals(val_close, val_high, val_low, val_vol, fe, se, tr_, ad)
        pf = vbt.Portfolio.from_signals(close=val_close, entries=e, exits=x,
                                        init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        os_sr = float(pf.sharpe_ratio(freq='D')); os_ret = float(pf.total_return())
        os_dd = float(pf.max_drawdown()); nt = len(pf.trades)
        os_wr = np.nan; os_pf = np.nan
        if nt > 0:
            tr = pf.trades.returns.values if hasattr(pf.trades.returns,'values') else np.array(pf.trades.returns)
            if tr.size:
                p=tr[tr>0]; n=tr[tr<0]
                os_wr=(len(p)/len(tr))*100
                os_pf=p.sum()/abs(n.sum()) if len(n) and abs(n.sum())>0 else np.inf
        vy = max((val_close.index[-1]-val_close.index[0]).days/365.25, 1e-9)
        oos_results.append({'Rank':len(oos_results)+1,'Fast':fe,'Slow':se,'Trail':tr_,'ADX':ad,
            'IS_Sharpe':float(st['sharpe_ratio']),'IS_Return':float(st['total_return']),
            'IS_Trades':int(st['total_trades']),'OOS_Sharpe':os_sr,'OOS_Return':os_ret,
            'OOS_MaxDD':os_dd,'OOS_Trades':nt,'OOS_WR':os_wr,'OOS_PF':os_pf,'OOS_T/Yr':nt/vy,
            'Degrad':((os_sr-st['sharpe_ratio'])/abs(st['sharpe_ratio'])*100) if st['sharpe_ratio']!=0 else np.nan})
    print(pd.DataFrame(oos_results).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    fig, axes = plt.subplots(1, min(5,len(oos_results)), figsize=(20,5), squeeze=False)
    for i, row in enumerate(oos_results):
        ax = axes[0][i]
        e, x = generate_v2_signals(val_close, val_high, val_low, val_vol, row['Fast'], row['Slow'], row['Trail'], row['ADX'])
        pf = vbt.Portfolio.from_signals(close=val_close, entries=e, exits=x, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        pf.value().plot(ax=ax, color='#2563EB')
        ax.set_title(f"#{row['Rank']} F={row['Fast']} S={row['Slow']}\nSR={row['OOS_Sharpe']:.2f} T={row['OOS_Trades']}", fontsize=9)
        ax.axhline(100_000, ls='--', color='gray', alpha=0.5)
    fig.suptitle(f"OOS Equity - {TICKER} EMA v2", fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

# Parameter Sensitivity Analysis
Vary each param around best value, hold others fixed. Color: green=better, red=worse vs base.

In [ ]:
# SENSITIVITY ANALYSIS
if results_df.empty:
    print("No results.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    fb=int(best['fast_ema']); sb=int(best['slow_ema']); tb=float(best['trail_atr_mult']); ab=int(best['adx_min'])
    print(f"Best: F={fb} S={sb} T={tb:.1f} ADX>={ab} | Sharpe={best['sharpe_ratio']:.3f}")

    cfgs = [('fast_ema',fb,list(range(max(3,fb-10),fb+11))),('slow_ema',sb,list(range(max(10,sb-15),sb+16,2))),
            ('trail_atr_mult',tb,[1.0,1.5,2.0,2.5,3.0,3.5]),('adx_min',ab,[10,15,20,25,30])]
    fig, axes = plt.subplots(2, 4, figsize=(20,10))
    for col, (pn, bv, vr) in enumerate(cfgs):
        for ri, (dl, dc, dh, dl2, dv) in enumerate([
            ("IS",train_close,train_high,train_low,train_vol),
            ("OOS",val_close,high_s.iloc[split_idx:],low_s.iloc[split_idx:],vol_s.iloc[split_idx:])]):
            ax = axes[ri][col]; sharpes = []
            for v in vr:
                kw = {'fast_ema':fb,'slow_ema':sb,'trail_atr_mult':tb,'adx_min':ab}; kw[pn]=v
                if kw['slow_ema']<=kw['fast_ema']+5: sharpes.append(np.nan); continue
                try:
                    e,x = generate_v2_signals(dc,dh,dl2,dv,**kw)
                    pf = vbt.Portfolio.from_signals(close=dc,entries=e,exits=x,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')
                    sharpes.append(float(pf.sharpe_ratio(freq='D')))
                except: sharpes.append(np.nan)
            bs = best['sharpe_ratio']
            colors = ['gray' if np.isnan(s) else ('#059669' if (s-bs)/abs(bs)*100>10 else '#86efac' if (s-bs)/abs(bs)*100>=0 else '#fb923c' if (s-bs)/abs(bs)*100>=-10 else '#DC2626') for s in sharpes]
            ax.bar(range(len(vr)),sharpes,color=colors)
            ax.set_xticks(range(len(vr))); ax.set_xticklabels([str(v) for v in vr],fontsize=7,rotation=45)
            bi = next((vi for vi,v in enumerate(vr) if v==bv or (isinstance(bv,float) and abs(v-bv)<0.01)),None)
            if bi is not None: ax.axvline(bi,ls='--',color='#2563EB',lw=1.5)
            ax.set_title(f"{dl}: {pn}",fontsize=10); ax.set_ylabel("Sharpe")
    fig.suptitle(f"Sensitivity - {TICKER} EMA v2",fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# FEATURE ABLATION - v1 vs v2
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fe=int(best['fast_ema']); se=int(best['slow_ema']); tr_=float(best['trail_atr_mult']); ad=int(best['adx_min'])

configs = [
    ("v1 Baseline", lambda c,h,l,v: generate_v1_signals(c, fe, se, se+20)),
    ("v2 No Pullback", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,tr_,ad,pullback=False)),
    ("v2 No ADX", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,tr_,adx_min=0)),
    ("v2 Tight (1.5)", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,1.5,ad)),
    ("v2 Wide (3.0)", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,3.0,ad)),
    ("v2 FULL", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,tr_,ad)),
]

print(f"ABLATION | Best: F={fe} S={se} T={tr_:.1f} ADX>={ad}")
print("=" * 90)
rows = []
for label, fn in configs:
    for sl, c, h, l, v in [("IS",train_close,train_high,train_low,train_vol),
                            ("OOS",val_close,high_s.iloc[split_idx:],low_s.iloc[split_idx:],vol_s.iloc[split_idx:])]:
        try:
            e, x = fn(c, h, l, v)
            pf = vbt.Portfolio.from_signals(close=c,entries=e,exits=x,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')
            yr = max((c.index[-1]-c.index[0]).days/365.25,1e-9)
            rows.append({'Config':label,'Split':sl,'Sharpe':float(pf.sharpe_ratio(freq='D')),
                'Return':float(pf.total_return()),'MaxDD':float(pf.max_drawdown()),
                'Trades':len(pf.trades),'T/Yr':len(pf.trades)/yr,
                'WR':float((pf.trades.returns.values>0).mean()*100) if len(pf.trades) else np.nan})
        except:
            rows.append({'Config':label,'Split':sl,'Sharpe':np.nan,'Return':np.nan,'MaxDD':np.nan,'Trades':0,'T/Yr':0,'WR':np.nan})

adf = pd.DataFrame(rows)
for s in ['IS','OOS']:
    print(f"\n  {s}:")
    print(adf[adf['Split']==s][['Config','Sharpe','Return','MaxDD','Trades','T/Yr','WR']].to_string(index=False,float_format=lambda x:f"{x:.3f}"))

In [ ]:
# MULTI-ASSET OOS
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fe=int(best['fast_ema']); se=int(best['slow_ema']); tr_=float(best['trail_atr_mult']); ad=int(best['adx_min'])
print(f"Testing across {len(all_data)} tickers | F={fe} S={se} T={tr_:.1f} ADX>={ad}")

multi = []; nt = len(all_data); cols = min(4,nt); rows = (nt+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols,4*rows), squeeze=False)
for ti,(tk,df) in enumerate(all_data.items()):
    c,h,l,v = get_ohlcv(df); sp = int(len(c)*TRAIN_RATIO)
    oc=c.iloc[sp:]; oh=h.iloc[sp:]; ol=l.iloc[sp:]; ov=v.iloc[sp:]
    try:
        e,x = generate_v2_signals(oc,oh,ol,ov,fe,se,tr_,ad)
        pf = vbt.Portfolio.from_signals(close=oc,entries=e,exits=x,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')
        yr = max((oc.index[-1]-oc.index[0]).days/365.25,1e-9)
        sr=float(pf.sharpe_ratio(freq='D')); rt=float(pf.total_return()); dd=float(pf.max_drawdown()); tn=len(pf.trades)
        multi.append({'Ticker':tk,'SR':sr,'Return':rt,'MaxDD':dd,'Trades':tn,'T/Yr':tn/yr})
        ax=axes[ti//cols][ti%cols]; pf.value().plot(ax=ax,color='#2563EB')
        ax.axhline(100_000,ls='--',color='gray',alpha=0.5); ax.set_title(f"{tk} SR={sr:.2f} T={tn}",fontsize=10)
    except:
        multi.append({'Ticker':tk,'SR':np.nan,'Return':np.nan,'MaxDD':np.nan,'Trades':0,'T/Yr':0})
        axes[ti//cols][ti%cols].set_title(f"{tk} ERROR")
for j in range(nt,rows*cols): axes[j//cols][j%cols].set_visible(False)
fig.suptitle("EMA v2 Multi-Asset OOS",fontsize=14,fontweight='bold'); plt.tight_layout(); plt.show()
mdf = pd.DataFrame(multi); print(mdf.to_string(index=False,float_format=lambda x:f"{x:.3f}"))
print(f"\nAvg SR: {mdf['SR'].mean():.3f} | Avg T/Yr: {mdf['T/Yr'].mean():.1f}")

In [ ]:
# FTMO MONTE CARLO - v1 vs v2
import json as _json
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fe=int(best['fast_ema']); se=int(best['slow_ema']); tr_=float(best['trail_atr_mult']); ad=int(best['adx_min'])
vh=high_s.iloc[split_idx:]; vl=low_s.iloc[split_idx:]; vv=vol_s.iloc[split_idx:]

e2,x2 = generate_v2_signals(val_close,vh,vl,vv,fe,se,tr_,ad)
pf2 = vbt.Portfolio.from_signals(close=val_close,entries=e2,exits=x2,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')
e1,x1 = generate_v1_signals(val_close,fe,se,se+20)
pf1 = vbt.Portfolio.from_signals(close=val_close,entries=e1,exits=x1,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')

try:
    with open('../config/ftmo_rules.json','r') as f: ftmo=_json.load(f)
except: ftmo={"account_size":100000,"profit_target_pct":10,"max_daily_loss_pct":5,"max_total_loss_pct":10}

ACC=ftmo['account_size']; TGT=ACC*ftmo['profit_target_pct']/100
MXD=ACC*ftmo['max_daily_loss_pct']/100; MXT=ACC*ftmo['max_total_loss_pct']/100
DAYS=30; NSIM=10_000

def ftmo_mc(rets_s, label):
    r = rets_s.dropna().values
    if len(r)<5: print(f"  {label}: too few"); return None
    ps=0; bd=0; bt=0; to=0; fb=[]
    for _ in range(NSIM):
        s=np.random.choice(r,DAYS,True); b=ACC; blown=False
        for rv in s:
            pnl=b*rv; b+=pnl
            if pnl<-MXD: bd+=1; blown=True; break
            if ACC-b>MXT: bt+=1; blown=True; break
        fb.append(b)
        if not blown:
            if b>=ACC+TGT: ps+=1
            else: to+=1
    pr=ps/NSIM*100
    print(f"  {label}: Pass={pr:.1f}% DailyDD={bd/NSIM*100:.1f}% TotalDD={bt/NSIM*100:.1f}% Timeout={to/NSIM*100:.1f}%")
    return {'pass_rate':pr,'blown_daily':bd/NSIM*100,'blown_total':bt/NSIM*100,'balances':fb}

print(f"FTMO MC | {NSIM:,} sims x {DAYS} days | Acct=${ACC:,} Target=+${TGT:,}")
print("=" * 70)
mc1 = ftmo_mc(pf1.returns(), "v1 (cross only)")
mc2 = ftmo_mc(pf2.returns(), "v2 (trail+pullback)")

if mc1 and mc2:
    fig,(a1,a2) = plt.subplots(1,2,figsize=(14,5))
    for mc,lb,cl,ax in [(mc1,'v1','#94a3b8',a1),(mc2,'v2','#2563EB',a2)]:
        ax.hist(mc['balances'],bins=80,color=cl,alpha=0.8,edgecolor='white',lw=0.3)
        ax.axvline(ACC+TGT,color='#059669',ls='--',lw=2,label=f"Target ${ACC+TGT:,}")
        ax.axvline(ACC-MXT,color='#DC2626',ls='--',lw=2,label=f"Blown ${ACC-MXT:,}")
        ax.set_title(f"{lb} Pass={mc['pass_rate']:.1f}%",fontsize=13,fontweight='bold')
        ax.legend(fontsize=9)
    fig.suptitle(f"FTMO MC - {TICKER} EMA v1 vs v2",fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f"\nv2 improvement: +{mc2['pass_rate']-mc1['pass_rate']:.1f}pp")

In [ ]:
# ================================================================
# Universal Export Cell -- Professional PDF Tearsheet + Data Files
# ================================================================
# Uses shared lib/UNIVERSAL_EXPORT_CELL_v2.py for consistent output
# across all strategy notebooks.

STRATEGY_NAME = "EMA_Crossover_v2"
PARAM_COLS = ['fast_ema', 'slow_ema', 'trail_atr_mult', 'adx_min']

import os
_lib_dir = 'lib' if os.path.isdir('lib') else os.path.join('..', 'lib')
exec(open(os.path.join(_lib_dir, 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())
